In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp

In [0]:
df=spark.read.format("delta")\
    .load("/Volumes/workspace/bronze/bronzevolume/passengers/data/")
df=df.drop("_rescued_data")\
        .withColumn('modification',current_timestamp())

display(df)

In [0]:
@dp.view(
    name="trans_passengers"
)
def trans_passengers():

    df=spark.readStream.format("delta")\
        .load("/Volumes/workspace/bronze/bronzevolume/passengers/data")
    return df

In [0]:

dp.create_streaming_table("silver_passengers")

dp.create_auto_cdc_flow(
  target = "silver_passengers",
  source = "trans_passengers",
  keys = ["passenger_id"],
  sequence_by = col("passenger_id"),
  stored_as_scd_type = 1
)